In [1]:
import json
from dataclasses import dataclass, field, asdict
import urllib.request

messages = [
    {"role": "system", "content": "Answer briefly and accurately."},
    {"role": "user", "content": "What is 2 + 2?"}
]

print(json.dumps(messages, indent=2))

context_limit = 4096
input_tokens = 3000
reserved_output = 500

print("Remaining budget:", context_limit - input_tokens - reserved_output)

[
  {
    "role": "system",
    "content": "Answer briefly and accurately."
  },
  {
    "role": "user",
    "content": "What is 2 + 2?"
  }
]
Remaining budget: 596


I created messages for the AI and calculated how many tokens are left for the response.

In [2]:
@dataclass
class Response:
    content: str = ""
    reasoning: str | None = None
    tool_call: dict | None = None
    metadata: dict = field(default_factory=dict)

example = Response(content="4")

print(example)
print(example.metadata)

Response(content='4', reasoning=None, tool_call=None, metadata={})
{}


I created a response class to organize the AI's answers and other information.

In [3]:
def parse_response(data):
    choices = data.get("choices") or []

    if not choices:
        raise ValueError("Endpoint returned no choices")

    message = choices[0]["message"]
    calls = message.get("tool_calls") or []

    return Response(
        content=message.get("content") or "",
        reasoning=message.get("reasoning"),
        tool_call=calls[0] if calls else None,
        metadata={
            "model": data.get("model", "unknown"),
            "usage": data.get("usage") or {}
        }
    )

sample = {"choices": [{"message": {"content": "4"}}]}

print(parse_response(sample))

Response(content='4', reasoning=None, tool_call=None, metadata={'model': 'unknown', 'usage': {}})


I made a function that reads AI responses and handles missing information.

In [4]:
def http_transport(url, body):
    request = urllib.request.Request(
        url,
        data=json.dumps(body).encode("utf-8"),
        headers={"Content-Type": "application/json"}
    )

    with urllib.request.urlopen(request, timeout=60) as reply:
        return json.loads(reply.read().decode("utf-8"))


class LLM:
    def __init__(self, model, transport=http_transport,
                 base_url="http://localhost:11434/v1"):
        self.model = model
        self.transport = transport
        self.base_url = base_url.rstrip("/")

    def generate(self, messages):
        body = {
            "model": self.model,
            "messages": messages,
            "stream": False
        }

        data = self.transport(
            self.base_url + "/chat/completions", body
        )

        return parse_response(data)

I created a wrapper that sends messages to an AI model and processes its responses.

In [5]:
def fixture_transport(url, body):
    scripted = {
        "What is 2 + 2?": "4",
        "What is 3 + 5?": "8"
    }

    task = body["messages"][-1]["content"]
    answer = scripted.get(task, "UNSCRIPTED: connect a real model.")

    return {
        "model": "SCRIPTED-FIXTURE",
        "choices": [{
            "message": {
                "role": "assistant",
                "content": answer
            }
        }],
        "usage": {
            "prompt_tokens": 20,
            "completion_tokens": 1
        }
    }

llm = LLM("classroom-fixture", transport=fixture_transport)

response = llm.generate(messages)

print(asdict(response))

assert response.content == "4"

{'content': '4', 'reasoning': None, 'tool_call': None, 'metadata': {'model': 'SCRIPTED-FIXTURE', 'usage': {'prompt_tokens': 20, 'completion_tokens': 1}}}


I tested the program using preset answers to make sure everything works without a real AI model.

In [6]:
@dataclass
class Step:
    thought: str = ""
    action: dict | None = None
    observation: str | None = None
    answer: str | None = None
    metadata: dict = field(default_factory=dict)


class Trajectory:
    def __init__(self):
        self.runs = []

    def initialize(self, query):
        self.runs.append({"query": query, "steps": []})

    def add(self, response):
        if not self.runs:
            raise RuntimeError("Initialize a run first")

        if response.tool_call is not None:
            raise NotImplementedError("Tools are a later lesson")

        step = Step(
            answer=response.content,
            metadata=dict(response.metadata)
        )

        self.runs[-1]["steps"].append(step)

I created a way to track the AI's answers and save each step for review.

In [7]:
class TinyAgent:
    def __init__(self, llm):
        self.llm = llm
        self.trajectory = Trajectory()

    def run(self, task):
        self.trajectory.initialize(task)

        messages = [
            {"role": "system", "content": "Answer briefly and accurately."},
            {"role": "user", "content": task}
        ]

        response = self.llm.generate(messages)

        self.trajectory.add(response)

        return response.content


agent = TinyAgent(llm)

print(agent.run("What is 2 + 2?"))
print(agent.trajectory.runs)

assert len(agent.trajectory.runs) == 1
assert agent.trajectory.runs[0]["steps"][0].answer == "4"

4
[{'query': 'What is 2 + 2?', 'steps': [Step(thought='', action=None, observation=None, answer='4', metadata={'model': 'SCRIPTED-FIXTURE', 'usage': {'prompt_tokens': 20, 'completion_tokens': 1}})]}]


I connected everything so the agent can answer questions and record its results.

In [8]:
print(agent.run("What is 3 + 5?"))

assert len(agent.trajectory.runs) == 2
assert all(len(run["steps"]) == 1 for run in agent.trajectory.runs)

assert agent.trajectory.runs[0]["steps"][0].answer == "4"
assert agent.trajectory.runs[1]["steps"][0].answer == "8"

try:
    parse_response({"choices": []})
except ValueError as error:
    print("Expected boundary error:", error)
else:
    raise AssertionError("Empty choices should fail")

print("Unlisted task:", agent.run("Name a planet."))

8
Expected boundary error: Endpoint returned no choices
Unlisted task: UNSCRIPTED: connect a real model.


I tested another question and checked how the program handles errors and unknown questions.

In [9]:
def updated_fixture(url, body):
    scripted = {
        "What is 2 + 2?": "4",
        "What is 3 + 5?": "8",
        "What is 10 + 5?": "15"
    }

    task = body["messages"][-1]["content"]

    return {
        "model": "SCRIPTED-FIXTURE",
        "choices": [{
            "message": {
                "content": scripted.get(task, "UNSCRIPTED")
            }
        }]
    }

test_agent = TinyAgent(
    LLM("classroom-fixture", transport=updated_fixture)
)

print(test_agent.run("What is 10 + 5?"))
print(test_agent.run("What is 2 + 2?"))
print(test_agent.run("What is 3 + 5?"))
print(test_agent.run("Name a planet."))

print(test_agent.trajectory.runs)

15
4
8
UNSCRIPTED
[{'query': 'What is 10 + 5?', 'steps': [Step(thought='', action=None, observation=None, answer='15', metadata={'model': 'SCRIPTED-FIXTURE', 'usage': {}})]}, {'query': 'What is 2 + 2?', 'steps': [Step(thought='', action=None, observation=None, answer='4', metadata={'model': 'SCRIPTED-FIXTURE', 'usage': {}})]}, {'query': 'What is 3 + 5?', 'steps': [Step(thought='', action=None, observation=None, answer='8', metadata={'model': 'SCRIPTED-FIXTURE', 'usage': {}})]}, {'query': 'Name a planet.', 'steps': [Step(thought='', action=None, observation=None, answer='UNSCRIPTED', metadata={'model': 'SCRIPTED-FIXTURE', 'usage': {}})]}]


I added another question and tested the agent again. The answers are preset, so this doesn't mean the AI actually learned anything. In this assignment, I learned how an AI agent sends messages, processes answers, and tracks results. I also learned that testing preset answers is different from using a real AI model. The agent can save results, but it doesn't automatically remember past conversations.